# Pokémon TCG AI — evaluate a trained checkpoint on Kaggle T4×2

Lightweight companion to `kaggle_notebook.ipynb`. Does **only** evaluation:

1. Pulls the repo from GitHub.
2. Mounts a training-artefact dataset (from a previous training session).
3. Loads the newest checkpoint under that dataset.
4. Runs three head-to-head matches on the stacked aggro deck:
   * A. Trained agent vs Random  (n=40)
   * B. Trained agent vs Heuristic-MCTS  (n=20)
   * C. Trained mirror  (n=10)
5. Writes CSVs, `summary.json`, and three figures under `/kaggle/working/`.

**Runtime:** ~30-90 min depending on `EVAL_AGENT_ITERATIONS` and how long games take to terminate.

**No training.** Safe to run any number of times.

**Prerequisites (do these once, in the Kaggle UI):**

- Settings → Accelerator: **GPU T4×2**.
- Settings → Internet: **On**.
- Add your training-artefact dataset (the one you published from the main notebook — e.g. `ptcg-train-v3`) via the right sidebar → *Add Data*.

## 0. Configuration — edit before running

In [ ]:
# === Where the trained checkpoint lives ===
# Path to the mounted Kaggle dataset containing the training run.
# Standard layout: <dataset root>/experiments/kaggle_t4x2_run/checkpoints/ckpt_XXXXXX.pt
CHECKPOINT_DATASET = "/kaggle/input/ptcg-train-v3"   # edit me

# Which checkpoint to evaluate. Options:
#   "latest"  — use the highest-numbered ckpt_*.pt (recommended)
#   "best"    — use the last-promoted checkpoint from manifest.json
#   "ckpt_XXXXXX"  — exact stem, e.g. "ckpt_008900"
CHECKPOINT_CHOICE = "latest"

# === Repository source ===
REPO_GIT_URL = "https://github.com/arnav-chauhan-kgpian/PTCG.git"
REPO_GIT_BRANCH = "main"

# === Evaluation knobs ===
EVAL_N_GAMES_AGENT_VS_RANDOM = 40
EVAL_N_GAMES_AGENT_VS_HEURISTIC = 20
EVAL_N_GAMES_MIRROR = 10
EVAL_MAX_ACTIONS_PER_GAME = 400
EVAL_AGENT_ITERATIONS = 32       # MCTS iter per decision

In [ ]:
import os, sys, subprocess, json, time, math, random, pathlib, statistics, csv
print("python:", sys.version.split()[0])
subprocess.run(["nvidia-smi", "-L"], check=False)
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "devices:", torch.cuda.device_count())

# Silence loguru DEBUG spam (the effect-matcher fires debug on every card play).
os.environ["POKEMON_AI_LOG_LEVEL"] = "ERROR"
try:
    from loguru import logger as _loguru
    _loguru.remove()
    _loguru.add(sys.stderr, level="ERROR")
    print("loguru sink: stderr @ ERROR")
except ImportError:
    pass

## 1. Get the repo

In [ ]:
REPO_DIR = "/kaggle/working/PTCG"
def have_repo():
    return (pathlib.Path(REPO_DIR) / "src" / "cli" / "main.py").exists()

if not have_repo():
    rc = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_GIT_BRANCH, REPO_GIT_URL, REPO_DIR],
        check=False,
    ).returncode
    assert rc == 0 and have_repo(), "git clone failed — enable Internet in Settings"

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install (uses Kaggle's pre-installed torch)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "-e", ".[dev,server]"], check=True)
print("repo installed at", REPO_DIR)

## 2. Locate the checkpoint

In [ ]:
def find_checkpoint(dataset_root: str, choice: str) -> pathlib.Path:
    root = pathlib.Path(dataset_root)
    assert root.exists(), (
        f"CHECKPOINT_DATASET={dataset_root} not found. Did you *Add Data* the training dataset?"
    )
    # Standard layout: <root>/experiments/<name>/checkpoints/ckpt_*.pt
    # But also accept: <root>/checkpoints/ckpt_*.pt or <root>/**/ckpt_*.pt
    candidates = []
    for pattern in [
        "experiments/*/checkpoints/ckpt_*.pt",
        "checkpoints/ckpt_*.pt",
        "**/ckpt_*.pt",
    ]:
        found = sorted(root.glob(pattern))
        if found:
            candidates = found
            break
    assert candidates, (
        f"No ckpt_*.pt found under {dataset_root}. Inspect the dataset layout:\n"
        + "\n".join(f"  {p}" for p in list(root.rglob('*.pt'))[:20])
    )

    if choice == "latest":
        return candidates[-1]
    if choice == "best":
        # Look for manifest.json alongside the checkpoints
        for manifest in root.rglob("manifest.json"):
            try:
                data = json.loads(manifest.read_text(encoding="utf-8"))
                best = data.get("best_checkpoint")
                if best:
                    for c in candidates:
                        if c.stem == best:
                            return c
            except Exception:
                continue
        print("⚠ no manifest.json best_checkpoint found; falling back to latest")
        return candidates[-1]
    # Explicit name
    for c in candidates:
        if c.stem == choice:
            return c
    raise FileNotFoundError(f"checkpoint {choice!r} not found; available: {[c.stem for c in candidates[-5:]]}")

CKPT_PATH = find_checkpoint(CHECKPOINT_DATASET, CHECKPOINT_CHOICE)
print(f"using checkpoint: {CKPT_PATH}")
print(f"  size: {CKPT_PATH.stat().st_size/1024/1024:.2f} MB")

# Sanity — inspect payload keys
payload = torch.load(str(CKPT_PATH), map_location="cpu", weights_only=True)
print(f"  payload keys: {list(payload.keys())}")

## 3. Build the stacked aggro deck + action map

Must match the deck used during training, otherwise the policy head's action_size will mismatch and evaluation fails.

In [ ]:
from src.cards import load_repository
from src.simulator import PokemonTCGSimulator
from src.mcts.node import MCTSAction

repo = load_repository()
print(f"repository: {len(repo.list_all())} cards")

def build_stacked_deck(repo):
    from src.cards.enums import Stage
    from src.cards.models import EnergyCard, PokemonCard
    cands = []
    for c in repo.list_all():
        if isinstance(c, PokemonCard) and c.stage == Stage.BASIC and c.attacks:
            for att in c.attacks:
                cost_n = att.cost.total_count if att.cost else 99
                if cost_n <= 2 and att.damage and att.damage.base >= 20:
                    cands.append((c, cost_n, att.damage.base))
                    break
    cands.sort(key=lambda x: (x[1], -x[2]))
    top5 = [c for c, _, _ in cands[:5]]
    deck = []
    for c in top5:
        deck.extend([c.card_id] * 4)
    energies = [c for c in repo.list_all() if isinstance(c, EnergyCard)]
    basic = next((e for e in energies if "Basic" in (e.name or "")), energies[0])
    deck.extend([basic.card_id] * (60 - len(deck)))
    return deck[:60], [c.name for c in top5], basic.name

DECK, TOP5, ENERGY = build_stacked_deck(repo)
print("deck:", len(DECK), "cards /", len(set(DECK)), "unique")
print("attackers:", TOP5, " energy:", ENERGY)

# Build the action map identically to the training notebook.
action_set = set()
for s in range(8):
    sim2 = PokemonTCGSimulator(repo, seed=s)
    st = sim2.start_game(DECK, DECK)
    for _ in range(64):
        legal = sim2.legal_actions(st)
        if not legal:
            break
        for a in legal:
            action_set.add((a.action_type, tuple(sorted(a.details.items())) if isinstance(a.details, dict) else a.details))
        st = sim2.apply_action(st, legal[0])
        if sim2.is_terminal(st):
            break
action_map = []
for at, details in sorted(action_set, key=lambda x: x[0]):
    details_dict = dict(details) if isinstance(details, tuple) else {}
    action_map.append(MCTSAction(action_type=at, details=details_dict))
print("action_map size:", len(action_map))

## 4. Build the three agents (trained / heuristic / trained-b)

In [ ]:
from src.competition.agent import AgentConfig, CompetitionAgent
from src.competition.inference_engine import InferenceEngine
from src.mcts.checkpoints import CheckpointManager

device = "cuda" if torch.cuda.is_available() else "cpu"

def build_trained_agent(ckpt_path: pathlib.Path):
    """Load a training-pipeline checkpoint into a CompetitionAgent."""
    mgr = CheckpointManager(ckpt_path.parent)
    network, _meta = mgr.load(ckpt_path.stem, device=device)
    inference = InferenceEngine(network)
    return CompetitionAgent(
        simulator=PokemonTCGSimulator(repo, seed=42),
        inference=inference,
        config=AgentConfig(iterations=EVAL_AGENT_ITERATIONS, time_budget_s=2.0),
    )

agent_trained   = build_trained_agent(CKPT_PATH)
agent_trained_b = build_trained_agent(CKPT_PATH)
agent_heur      = CompetitionAgent.load(
    None, repository=repo,
    config=AgentConfig(iterations=EVAL_AGENT_ITERATIONS, time_budget_s=2.0),
)

print("agents constructed")
print("  trained.inference is not None:", agent_trained.inference is not None)
print("  heur.inference is not None:   ", agent_heur.inference is not None)
print("  device:", device)

## 5. Run the head-to-head evaluation

In [ ]:
from src.game_state.zones import GameStatus

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0.0, c-h), min(1.0, c+h))

def play(sim, deck, pa, pb, max_actions, seed):
    rng = random.Random(seed)
    state = sim.start_game(deck, deck)
    n_actions, lats = 0, []
    while n_actions < max_actions and not sim.is_terminal(state):
        legal = sim.legal_actions(state)
        if not legal:
            break
        pid = state.current_player
        t0 = time.perf_counter()
        a = (pa if pid == 0 else pb)(state, legal, rng)
        if pid == 0:
            lats.append((time.perf_counter() - t0) * 1000.0)
        state = sim.apply_action(state, a)
        n_actions += 1
    winner = (
        0 if state.game_status == GameStatus.PLAYER_0_WIN
        else 1 if state.game_status == GameStatus.PLAYER_1_WIN
        else "draw" if state.game_status == GameStatus.DRAW
        else "timeout"
    )
    return dict(
        winner=winner, actions=n_actions, final_turn=state.turn_number,
        p0_prizes_left=state.players[0].prizes_remaining,
        p1_prizes_left=state.players[1].prizes_remaining,
        knockouts=len(state.knockout_history),
        avg_ms_p0=round(statistics.mean(lats), 2) if lats else 0.0,
        max_ms_p0=round(max(lats), 2) if lats else 0.0,
    )

def random_pol(state, legal, rng):
    return rng.choice(legal)

def make_agent_pol(agent):
    def pol(state, legal, rng):
        a = agent.choose_action(state)
        return a if a is not None else rng.choice(legal)
    return pol

def run_match(label, pa, pb, n_games):
    print(f"\n=== {label}  n={n_games}", flush=True)
    rows = []
    t_start = time.perf_counter()
    for i in range(n_games):
        sim = PokemonTCGSimulator(repo, seed=500 + i)
        r = play(sim, DECK, pa, pb, EVAL_MAX_ACTIONS_PER_GAME, seed=600 + i)
        rows.append({"game_id": i, **r})
        elapsed = time.perf_counter() - t_start
        eta_min = (elapsed / (i+1)) * (n_games - i - 1) / 60
        print(f"  g{i:>2}/{n_games}: winner={r['winner']!s:<7} actions={r['actions']:>3} "
              f"prizes=({r['p0_prizes_left']},{r['p1_prizes_left']}) kos={r['knockouts']} "
              f"avg_ms={r['avg_ms_p0']:>6.1f}  eta={eta_min:.1f}min", flush=True)
    return rows

def summarize(label, rows):
    n = len(rows)
    p0 = sum(1 for r in rows if r["winner"] == 0)
    p1 = sum(1 for r in rows if r["winner"] == 1)
    d  = sum(1 for r in rows if r["winner"] == "draw")
    t  = sum(1 for r in rows if r["winner"] == "timeout")
    lo, hi = wilson_ci(p0, n)
    return dict(
        label=label, n_games=n,
        p0_wins=p0, p1_wins=p1, draws=d, timeouts=t,
        p0_win_rate=round(p0/n if n else 0, 4),
        p0_win_rate_wilson_95_ci=[round(lo,4), round(hi,4)],
        termination_rate=round(sum(1 for r in rows if r["winner"]!="timeout")/n, 4) if n else 0,
        avg_actions_per_game=round(statistics.mean(r["actions"] for r in rows), 1) if rows else 0,
        avg_knockouts_per_game=round(statistics.mean(r["knockouts"] for r in rows), 1) if rows else 0,
        avg_decision_ms_p0=round(statistics.mean(r["avg_ms_p0"] for r in rows if r["avg_ms_p0"]>0), 1) if any(r["avg_ms_p0"]>0 for r in rows) else 0,
    )

OUT = pathlib.Path("/kaggle/working/eval")
OUT.mkdir(parents=True, exist_ok=True)

rows_ar     = run_match("A: Trained Agent vs Random",
                          make_agent_pol(agent_trained), random_pol,
                          EVAL_N_GAMES_AGENT_VS_RANDOM)
rows_ah     = run_match("B: Trained Agent vs Heuristic",
                          make_agent_pol(agent_trained), make_agent_pol(agent_heur),
                          EVAL_N_GAMES_AGENT_VS_HEURISTIC)
rows_mirror = run_match("C: Trained vs Trained (mirror)",
                          make_agent_pol(agent_trained), make_agent_pol(agent_trained_b),
                          EVAL_N_GAMES_MIRROR)

summaries = {
    "deck": {"top_attackers": TOP5, "basic_energy": ENERGY,
             "pokemon_count": 20, "energy_count": 40},
    "checkpoint": str(CKPT_PATH),
    "config": {
        "max_actions_per_game": EVAL_MAX_ACTIONS_PER_GAME,
        "agent_iterations": EVAL_AGENT_ITERATIONS,
        "device": device,
    },
    "matches": [
        summarize("trained_vs_random",         rows_ar),
        summarize("trained_vs_heuristic",      rows_ah),
        summarize("trained_vs_trained_mirror", rows_mirror),
    ],
}

for name, rows in [("trained_vs_random",     rows_ar),
                    ("trained_vs_heuristic",  rows_ah),
                    ("trained_vs_trained_mirror", rows_mirror)]:
    if not rows:
        continue
    with (OUT / f"{name}.csv").open("w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
(OUT / "summary.json").write_text(json.dumps(summaries, indent=2, default=str))

print("\nSUMMARY:")
for s in summaries["matches"]:
    print(f"  {s['label']:<28} p0_win={s['p0_win_rate']:.1%}  "
          f"CI={s['p0_win_rate_wilson_95_ci']}  "
          f"term={s['termination_rate']:.0%}  "
          f"avg_act={s['avg_actions_per_game']}")

## 6. Figures + packaging

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = pathlib.Path("/kaggle/working/figures"); FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160})

labels = [s["label"] for s in summaries["matches"]]
rates  = [s["p0_win_rate"] for s in summaries["matches"]]
lows   = [r - s["p0_win_rate_wilson_95_ci"][0] for r, s in zip(rates, summaries["matches"])]
highs  = [s["p0_win_rate_wilson_95_ci"][1] - r for r, s in zip(rates, summaries["matches"])]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(labels, rates, xerr=[lows, highs], color=["#5E5CFF","#FF647C","#FFB623"], capsize=5)
ax.axvline(0.5, color="#666", linestyle="--", linewidth=1, alpha=0.7, label="fair-coin (50%)")
ax.set_xlim(0, 1.0); ax.set_xlabel("p0 win rate (Wilson 95% CI)")
ax.set_title(f"Trained agent — head-to-head  (ckpt={CKPT_PATH.stem})")
for i, (r, s) in enumerate(zip(rates, summaries["matches"])):
    ax.text(r + max(highs[i], 0.02) + 0.02, i,
            f"{r:.1%}  n={s['n_games']}", va="center", fontsize=9)
ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(FIG/"win_rates.png", bbox_inches="tight"); plt.close()

term = [s["termination_rate"] for s in summaries["matches"]]
avga = [s["avg_actions_per_game"] for s in summaries["matches"]]
fig, ax = plt.subplots(figsize=(9, 4))
x = list(range(len(labels))); w = 0.4
b1 = ax.bar([i - w/2 for i in x], term, w, color="#5E5CFF", label="termination rate")
ax2 = ax.twinx()
b2 = ax2.bar([i + w/2 for i in x], avga, w, color="#FF647C", label="avg actions / game")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=10)
ax.set_ylim(0, 1.0); ax.set_ylabel("termination rate")
ax2.set_ylabel("average actions per game")
ax.set_title("Game completion — terminations and action counts")
l1,ll1 = ax.get_legend_handles_labels(); l2,ll2 = ax2.get_legend_handles_labels()
ax.legend(l1+l2, ll1+ll2, loc="upper left", fontsize=9)
plt.tight_layout(); fig.savefig(FIG/"termination_kos.png", bbox_inches="tight"); plt.close()

print("figures saved:", list(FIG.iterdir()))

# Package everything for download
SUBMIT = pathlib.Path("/kaggle/working/eval_bundle"); SUBMIT.mkdir(exist_ok=True)
subprocess.run(["cp", "-r", str(OUT), str(SUBMIT / "eval")], check=False)
subprocess.run(["cp", "-r", str(FIG), str(SUBMIT / "figures")], check=False)
subprocess.run(["cp", str(CKPT_PATH), str(SUBMIT / CKPT_PATH.name)], check=False)
(SUBMIT / "MANIFEST.json").write_text(json.dumps({
    "source_checkpoint": str(CKPT_PATH),
    "summary": summaries,
    "environment": {
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
        "device_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    },
}, indent=2, default=str))

print("\nContents of /kaggle/working/eval_bundle:")
subprocess.run(["ls", "-laR", str(SUBMIT)])